# Test In-Silico Results

In [1]:
import os
from collections import defaultdict, Counter

def parse_sequence_file(file_path):
    _, file_extension = os.path.splitext(file_path)

    if file_extension.lower() not in ['.fasta', '.fa', '.txt']:
        raise ValueError(f"Unsupported file extension: {file_extension}")

    def sequence_generator():
        current_id = None
        current_seq = []
        with open(file_path, 'r') as file:
            for line in file:
                line = line.strip()
                if not line:  # Skip empty lines
                    continue
                if line.startswith('>'):
                    if current_id:
                        seq = ''.join(current_seq).strip()
                        if seq:
                            yield current_id, seq
                    current_id = line[1:] if file_extension.lower() in ['.fasta', '.fa'] else ';'.join(line[1:].split(';')[:2])
                    current_seq = []
                else:
                    current_seq.append(line)
        if current_id:
            seq = ''.join(current_seq).strip()
            if seq:
                yield current_id, seq

    return sequence_generator()

def process_sequences(generator):
    total_sequences = 0
    unique_ids = set()
    unique_sequences = set()
    id_to_seq = {}
    seq_lengths = []

    for seq_id, seq in generator:
        total_sequences += 1
        unique_ids.add(seq_id)
        unique_sequences.add(seq)
        id_to_seq[seq_id] = seq
        seq_lengths.append(len(seq))

        # if total_sequences % 100000 == 0:
        #     print(f"Processed {total_sequences} sequences...")

    return {
        'total': total_sequences,
        'unique_ids': unique_ids,
        'unique_sequences': unique_sequences,
        'id_to_seq': id_to_seq,
        'seq_lengths': seq_lengths
    }

def check_intersection_between_results(folderA, folderB):
    filesA = [f for f in os.listdir(folderA) if f.endswith(('.fasta', '.fa', '.txt'))]
    filesB = [f for f in os.listdir(folderB) if f.endswith(('.fasta', '.fa', '.txt'))]

    base_filesA = {os.path.splitext(f)[0]: f for f in filesA}
    base_filesB = {os.path.splitext(f)[0]: f for f in filesB}

    files_same_base_name = set(base_filesA.keys()) & set(base_filesB.keys())

    if not files_same_base_name:
        print("No matching files found between the two folders. Check folder direcotry and content before re-running.")
        return

    results = defaultdict(dict)

    for file_name in files_same_base_name:
        fileA = base_filesA[file_name]
        fileB = base_filesB[file_name]
        pathA = os.path.join(folderA, fileA)
        pathB = os.path.join(folderB, fileB)

        try:
            # print(f"Processing {fileA} from Folder A...")
            recordsA = parse_sequence_file(pathA)
            resultsA = process_sequences(recordsA)

            # print(f"Processing {fileB} from Folder B...")
            recordsB = parse_sequence_file(pathB)
            resultsB = process_sequences(recordsB)
        except Exception as e:
            print(f"Error reading files: {fileA} and {fileB}, Error: {e}")
            continue

        idsA = resultsA['unique_ids']
        idsB = resultsB['unique_ids']
        sequencesA = resultsA['unique_sequences']
        sequencesB = resultsB['unique_sequences']

        repeated_headersA = resultsA['total'] - len(idsA)
        repeated_headersB = resultsB['total'] - len(idsB)
        repeated_sequencesA = resultsA['total'] - len(sequencesA)
        repeated_sequencesB = resultsB['total'] - len(sequencesB)

        sequence_count_diff = resultsB['total'] - resultsA['total']

        avg_lengthA = sum(resultsA['seq_lengths']) / len(resultsA['seq_lengths']) if resultsA['seq_lengths'] else 0
        avg_lengthB = sum(resultsB['seq_lengths']) / len(resultsB['seq_lengths']) if resultsB['seq_lengths'] else 0

        min_lengthA = min(resultsA['seq_lengths']) if resultsA['seq_lengths'] else 0
        max_lengthA = max(resultsA['seq_lengths']) if resultsA['seq_lengths'] else 0
        min_lengthB = min(resultsB['seq_lengths']) if resultsB['seq_lengths'] else 0
        max_lengthB = max(resultsB['seq_lengths']) if resultsB['seq_lengths'] else 0

        avg_diff = round(abs(avg_lengthA - avg_lengthB), 2)

        intersection = idsA & idsB
        only_in_A = idsA - idsB
        only_in_B = idsB - idsA

        all_sequences_in_B = sequencesA.issubset(sequencesB)
        all_headers_in_B = idsA.issubset(idsB)

        results[file_name] = {
            "total_A": resultsA['total'],
            "avg_lenA": round(avg_lengthA, 2),
            "min_lenA": min_lengthA,
            "max_lenA": max_lengthA,
            "total_B": resultsB['total'],
            "avg_lenB": round(avg_lengthB, 2),
            "min_lenB": min_lengthB,
            "max_lenB": max_lengthB,
            "intersection": len(intersection),
            "only_in_A": len(only_in_A),
            "only_in_B": len(only_in_B),
            "avg_diff": avg_diff,
            "repeated_headersA": repeated_headersA,
            "repeated_headersB": repeated_headersB,
            "repeated_sequencesA": repeated_sequencesA,
            "repeated_sequencesB": repeated_sequencesB,
            "sequence_count_diff": sequence_count_diff,
            "headers_only_in_A": list(only_in_A)
        }

        print(f"Comparison for {file_name}:")
        print(f"  Total sequences in {file_name} from Folder A: {results[file_name]['total_A']}")
        print(f"  Total sequences in {file_name} from Folder B: {results[file_name]['total_B']}")
        print(f"  Difference in sequence count (B - A): {results[file_name]['sequence_count_diff']}")
        print(f"  Intersection (Folder A and B): {results[file_name]['intersection']}")
        print(f"  Minimum Length of sequences in Folder A: {results[file_name]['min_lenA']}")
        print(f"  Maximum Length of sequences in Folder A: {results[file_name]['max_lenA']}")
        print(f"  Minimum Length of sequences in Folder B: {results[file_name]['min_lenB']}")
        print(f"  Maximum Length of sequences in Folder B: {results[file_name]['max_lenB']}")
        print(f"  Repeated headers in File A: {results[file_name]['repeated_headersA']}")
        print(f"  Repeated headers in File B: {results[file_name]['repeated_headersB']}")
        print(f"  Repeated sequences in File A: {results[file_name]['repeated_sequencesA']}")
        print(f"  Repeated sequences in File B: {results[file_name]['repeated_sequencesB']}")
        print(f"  Number of headers only in File A: {results[file_name]['only_in_A']}")
        print(f"  Number of headers only in File B: {results[file_name]['only_in_B']}")

        if all_sequences_in_B:
            print("All sequences from A are present in B.")
        else:
            print("Not all sequences from A are present in B.")

        if all_headers_in_B:
            print("All headers from A are present in B.")
        else:
            print("Not all headers from A are present in B.")

        print("-" * 60)

## Cutadapt 3 & 5

### New Script
Same sequence IDs are present in both but sequences are different as different number of allowed mismatches will haev different scores.

In [17]:
folder_cutadapt5 = "../../data/output_data/test-filter-relaxed/all_barcodes_w_pbr"
folder_cutadapt3 = "../../data/output_data/test-filter-relaxed/insert"

check_intersection_between_results(folder_cutadapt3, folder_cutadapt5)

Comparing rbcL_708F3_R3-1.txt:
Comparison for rbcL_708F3_R3-1.txt:
  Total sequences in rbcL_708F3_R3-1.txt from Folder A: 4440
  Total sequences in rbcL_708F3_R3-1.txt from Folder B: 5053
  Difference in sequence count (B - A): 613
  Intersection (Folder A and B): 4440
  Minimum Length of sequences in Folder A: 178
  Maximum Length of sequences in Folder A: 263
  Minimum Length of sequences in Folder B: 178
  Maximum Length of sequences in Folder B: 263
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Number of headers only in File A: 0
  Number of headers only in File B: 613
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparing rbcL_708F1_R3-2.txt:
Comparison for rbcL_708F1_R3-2.txt:
  Total sequences in rbcL_708F1_R3-2.txt from Folder A: 4769
  Total sequences in rbcL_708F1_R3-2.txt from Folder B: 5053
  Difference in sequence count (B - A): 284
  Intersection (Folder A and 

## Cutadapt & PGA

### New Script

In [20]:
folder_pga1 = "../../data/output_data/test-decimal-prcnt-95percid/amplicon/filtered"
folder_pga2 = "../../data/output_data/test-decimal-prcnt-95percid/pga"

check_intersection_between_results(folder_pga1, folder_pga2)

Comparing 12S_12S-V5-c.txt:
Comparison for 12S_12S-V5-c.txt:
  Total sequences in 12S_12S-V5-c.txt from Folder A: 716
  Total sequences in 12S_12S-V5-c.txt from Folder B: 745
  Difference in sequence count (B - A): 29
  Intersection (Folder A and B): 716
  Minimum Length of sequences in Folder A: 117
  Maximum Length of sequences in Folder A: 119
  Minimum Length of sequences in Folder B: 117
  Maximum Length of sequences in Folder B: 119
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Number of headers only in File A: 0
  Number of headers only in File B: 29
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparing 12S_Hauck2019a.txt:
Comparison for 12S_Hauck2019a.txt:
  Total sequences in 12S_Hauck2019a.txt from Folder A: 526
  Total sequences in 12S_Hauck2019a.txt from Folder B: 1684
  Difference in sequence count (B - A): 1158
  Intersection (Folder A and B): 526
  Minimum Len

In [21]:
folder_pga1 = "../../data/output_data/test-whole-prcnt-95percid/amplicon/filtered"
folder_pga2 = "../../data/output_data/test-whole-prcnt-95percid/pga"

check_intersection_between_results(folder_pga1, folder_pga2)

Comparing 12S_12S-V5-c.txt:
Comparison for 12S_12S-V5-c.txt:
  Total sequences in 12S_12S-V5-c.txt from Folder A: 716
  Total sequences in 12S_12S-V5-c.txt from Folder B: 745
  Difference in sequence count (B - A): 29
  Intersection (Folder A and B): 716
  Minimum Length of sequences in Folder A: 117
  Maximum Length of sequences in Folder A: 119
  Minimum Length of sequences in Folder B: 117
  Maximum Length of sequences in Folder B: 119
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Number of headers only in File A: 0
  Number of headers only in File B: 29
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparing 12S_Hauck2019a.txt:
Comparison for 12S_Hauck2019a.txt:
  Total sequences in 12S_Hauck2019a.txt from Folder A: 526
  Total sequences in 12S_Hauck2019a.txt from Folder B: 564
  Difference in sequence count (B - A): 38
  Intersection (Folder A and B): 526
  Minimum Length

In [17]:
folder_pga_txt = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/test-whole-prcnt-95percid/pga"
folder_pga_fasta = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/test-fasta/pga"

check_intersection_between_results(folder_pga_txt, folder_pga_fasta)

Comparison for 12S_12S-V5-a:
  Total sequences in 12S_12S-V5-a from Folder A: 832
  Total sequences in 12S_12S-V5-a from Folder B: 835
  Difference in sequence count (B - A): 3
  Intersection (Folder A and B): 832
  Minimum Length of sequences in Folder A: 140
  Maximum Length of sequences in Folder A: 146
  Minimum Length of sequences in Folder B: 140
  Maximum Length of sequences in Folder B: 146
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Number of headers only in File A: 0
  Number of headers only in File B: 3
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparison for 12S_Fuller1998:
  Total sequences in 12S_Fuller1998 from Folder A: 0
  Total sequences in 12S_Fuller1998 from Folder B: 0
  Difference in sequence count (B - A): 0
  Intersection (Folder A and B): 0
  Minimum Length of sequences in Folder A: 0
  Maximum Length of sequences in Folder A: 0
  Minimum Length 

In [19]:
folder_pga_no_hits_param = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/test-percid75-cov99-nomaxaccepts-filteredoutputs/pga"
folder_pga_maxhits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/test-percid75-cov99-maxaccepts10-filteredoutpts/pga"

check_intersection_between_results(folder_pga_no_hits_param, folder_pga_maxhits)

Comparison for 12S_12S-V5-a:
  Total sequences in 12S_12S-V5-a from Folder A: 834
  Total sequences in 12S_12S-V5-a from Folder B: 834
  Difference in sequence count (B - A): 0
  Intersection (Folder A and B): 834
  Minimum Length of sequences in Folder A: 140
  Maximum Length of sequences in Folder A: 146
  Minimum Length of sequences in Folder B: 140
  Maximum Length of sequences in Folder B: 146
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Number of headers only in File A: 0
  Number of headers only in File B: 0
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparison for 12S_Fuller1998:
  Total sequences in 12S_Fuller1998 from Folder A: 0
  Total sequences in 12S_Fuller1998 from Folder B: 0
  Difference in sequence count (B - A): 0
  Intersection (Folder A and B): 0
  Minimum Length of sequences in Folder A: 0
  Maximum Length of sequences in Folder A: 0
  Minimum Length 

### Diat.barcode
Percid: 75
Cov: 99
All hits vs top hits

In [12]:
folder_pga_all_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/test-percid75-cov99-maxaccmaxrej-0-diat.barcode/pga"
folder_pga_best_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/test-percid75-cov99-nomaxaccmaxrej-diat.barcode/pga"

check_intersection_between_results(folder_pga_best_hits, folder_pga_all_hits)

Comparison for rbcL_708F2_R3-1:
  Total sequences in rbcL_708F2_R3-1 from Folder A: 5072
  Total sequences in rbcL_708F2_R3-1 from Folder B: 783546
  Difference in sequence count (B - A): 778474
  Intersection (Folder A and B): 5072
  Minimum Length of sequences in Folder A: 227
  Maximum Length of sequences in Folder A: 328
  Minimum Length of sequences in Folder B: 227
  Maximum Length of sequences in Folder B: 344
  Repeated headers in File A: 0
  Repeated headers in File B: 778440
  Number of headers only in File A: 0
  Number of headers only in File B: 34
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparison for rbcL_708F1_R3-2:
  Total sequences in rbcL_708F1_R3-2 from Folder A: 5072
  Total sequences in rbcL_708F1_R3-2 from Folder B: 1447081
  Difference in sequence count (B - A): 1442009
  Intersection (Folder A and B): 5072
  Minimum Length of sequences in Folder A: 227
  Maximum Leng

In [18]:
folder_pga_all_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-allhits/pga/filtered"
folder_pga_best_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-besthits/pga/filtered"
folder_pga_20_best_hits = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-20besthits/pga/filtered"

check_intersection_between_results(folder_pga_best_hits, folder_pga_all_hits)

Comparison for rbcL_708F2_R3-1:
  Total sequences in rbcL_708F2_R3-1 from Folder A: 5374
  Total sequences in rbcL_708F2_R3-1 from Folder B: 2263562
  Difference in sequence count (B - A): 2258188
  Intersection (Folder A and B): 5374
  Minimum Length of sequences in Folder A: 178
  Maximum Length of sequences in Folder A: 266
  Minimum Length of sequences in Folder B: 178
  Maximum Length of sequences in Folder B: 266
  Repeated headers in File A: 0
  Repeated headers in File B: 2258177
  Repeated sequences in File A: 2978
  Repeated sequences in File B: 2260513
  Number of headers only in File A: 0
  Number of headers only in File B: 11
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparison for rbcL_708F1_R3-2:
  Total sequences in rbcL_708F1_R3-2 from Folder A: 5374
  Total sequences in rbcL_708F1_R3-2 from Folder B: 2883287
  Difference in sequence count (B - A): 2877913
  Intersection (Fol

In [8]:
folder_pga_all_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-allhits/pga/filtered"
folder_pga_best_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-besthit/pga/filtered"
folder_pga_20_best_hits_12s = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-20besthits/pga/filtered"

check_intersection_between_results(folder_pga_best_hits_12S, folder_pga_20_best_hits_12s)

Comparison for 12S_Hauck2019a:
  Total sequences in 12S_Hauck2019a from Folder A: 614
  Total sequences in 12S_Hauck2019a from Folder B: 2912
  Difference in sequence count (B - A): 2298
  Intersection (Folder A and B): 614
  Minimum Length of sequences in Folder A: 198
  Maximum Length of sequences in Folder A: 212
  Minimum Length of sequences in Folder B: 198
  Maximum Length of sequences in Folder B: 216
  Repeated headers in File A: 0
  Repeated headers in File B: 2164
  Repeated sequences in File A: 392
  Repeated sequences in File B: 2579
  Number of headers only in File A: 0
  Number of headers only in File B: 134
All sequences from A are present in B.
All headers from A are present in B.
------------------------------------------------------------
Comparison for 12S_MarVer1:
  Total sequences in 12S_MarVer1 from Folder A: 756
  Total sequences in 12S_MarVer1 from Folder B: 991
  Difference in sequence count (B - A): 235
  Intersection (Folder A and B): 756
  Minimum Length of 

In [9]:
folder_pga_best_hits_12S = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/id75-cov99-besthit/pga/filtered"
folder_cutadapt5 = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/12S/cutadapt5/all_barcodes_w_pbr/filtered"

check_intersection_between_results(folder_cutadapt5, folder_pga_best_hits_12S)

Comparison for 12S_Hauck2019a:
  Total sequences in 12S_Hauck2019a from Folder A: 762
  Total sequences in 12S_Hauck2019a from Folder B: 614
  Difference in sequence count (B - A): -148
  Intersection (Folder A and B): 608
  Minimum Length of sequences in Folder A: 198
  Maximum Length of sequences in Folder A: 212
  Minimum Length of sequences in Folder B: 198
  Maximum Length of sequences in Folder B: 212
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Repeated sequences in File A: 509
  Repeated sequences in File B: 392
  Number of headers only in File A: 154
  Number of headers only in File B: 6
Not all sequences from A are present in B.
Not all headers from A are present in B.
------------------------------------------------------------
Comparison for 12S_MarVer1:
  Total sequences in 12S_MarVer1 from Folder A: 736
  Total sequences in 12S_MarVer1 from Folder B: 756
  Difference in sequence count (B - A): 20
  Intersection (Folder A and B): 736
  Minimum Length o

In [10]:
folder_pga_best_hits_diat = "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/inserts/id75-cov99-besthits/pga/filtered"
folder_cutadapt5_diat= "/home/camilababo/Documents/coding-projects/DNAquaIMG-tool/DNAquaIMG/data/output_data/diat-barcode/cutadapt5/all_barcodes_w_pbr/filtered"

check_intersection_between_results(folder_cutadapt5_diat, folder_pga_best_hits_diat)

Comparison for rbcL_708F1_R3-1:
  Total sequences in rbcL_708F1_R3-1 from Folder A: 5054
  Total sequences in rbcL_708F1_R3-1 from Folder B: 5374
  Difference in sequence count (B - A): 320
  Intersection (Folder A and B): 4773
  Minimum Length of sequences in Folder A: 178
  Maximum Length of sequences in Folder A: 263
  Minimum Length of sequences in Folder B: 178
  Maximum Length of sequences in Folder B: 266
  Repeated headers in File A: 0
  Repeated headers in File B: 0
  Repeated sequences in File A: 2832
  Repeated sequences in File B: 2978
  Number of headers only in File A: 281
  Number of headers only in File B: 601
All sequences from A are present in B.
Not all headers from A are present in B.
------------------------------------------------------------
Comparison for rbcL_708F-DEG_R3-DEG:
  Total sequences in rbcL_708F-DEG_R3-DEG from Folder A: 5067
  Total sequences in rbcL_708F-DEG_R3-DEG from Folder B: 5374
  Difference in sequence count (B - A): 307
  Intersection (Fold

## Side Transformations

### Fix Paths for Previous Version

In [ ]:
import os

def rename_file_extensions(folder, old_extension, new_extension):
    for filename in os.listdir(folder):
        if filename.endswith(old_extension):
            base_name = os.path.splitext(filename)[0]
            new_filename = f"{base_name}{new_extension}"

            old_file = os.path.join(folder, filename)
            new_file = os.path.join(folder, new_filename)

            os.rename(old_file, new_file)
            print(f"Renamed: {old_file} -> {new_file}")

folder1 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/test_results/cutadapt3"
folder2 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/test_results/cutadapt5"
old_extension1 = ".cutadapt"
old_extension2 = ".maxerror"
new_extension = ".txt"

rename_file_extensions(folder1, old_extension1, new_extension)
rename_file_extensions(folder2, old_extension2, new_extension)

In [ ]:
folder3 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trialfolder_pga_fasta/cutadapt/12S/pga"
rename_file_extensions(folder3, ".pga", ".txt")

In [ ]:
folder1 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/cutadapt3"
folder2 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/cutadapt5"
folder3 = "/home/camilababo/Documents/DNAquaIMG/joana_scripts/Trial/cutadapt/12S/pga"
old_extension1 = ".cutadapt"
old_extension2 = ".maxerror"
old_extension3 = ".pga"
new_extension = ".txt"

#rename_file_extensions(folder1, old_extension1, new_extension)
rename_file_extensions(folder2, old_extension2, new_extension)
#rename_file_extensions(folder3, old_extension3, new_extension)